# Fine-tune intfloat/multilingual-e5-base — Vietnamese Product Search

Notebook fine-tune **`intfloat/multilingual-e5-base`** cho bài toán truy hồi sản phẩm tiếng Việt.

Output: `embedding_project/models/e5_base_finetuned_final/`

> Khuyến nghị: dùng GPU Colab (T4/V100/A100). E5-base nặng hơn MiniLM nhưng nhẹ hơn BGE-M3.

Model tham chiếu để đánh giá pretrained: `embedding_project/notebooks/evaluate_e5_pretrained_colab.ipynb`

## 1) Cài thư viện

In [ ]:
!pip -q install "sentence-transformers>=3.0.0" "transformers>=4.40.0" torch datasets pandas scikit-learn numpy tqdm

## 2) Clone repo và kiểm tra GPU

In [ ]:
import os
import shutil
import subprocess
import sys
import torch

GITHUB_REPO_URL = "https://github.com/your-username/your-repo.git"
REPO_DIR = "/content/llm_provider_benchmarking"

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", GITHUB_REPO_URL, REPO_DIR], check=True)

SCRIPTS_DIR = f"{REPO_DIR}/embedding_project/scripts"
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)

print('REPO_DIR:', REPO_DIR)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 3) Cấu hình preset E5-base

In [ ]:
from pathlib import Path
from model_presets import get_preset

PRESET = get_preset('e5-base')
PROJECT_ROOT = Path(REPO_DIR) / 'embedding_project'
DATA_DIR = PROJECT_ROOT / 'data'
MODELS_DIR = PROJECT_ROOT / 'models'
OUTPUT_EVAL_DIR = PROJECT_ROOT / 'outputs' / 'evaluation'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_EVAL_DIR.mkdir(parents=True, exist_ok=True)

USE_GPU = torch.cuda.is_available()
EPOCHS = PRESET.epochs
BATCH_SIZE = 8 if USE_GPU else 2
FP16 = USE_GPU
MAX_SEQ_LENGTH = PRESET.max_seq_length
FINAL_DIR = MODELS_DIR / PRESET.final_subdir

print('Base model:', PRESET.base_model)
print('Final dir:', FINAL_DIR)
print('epochs:', EPOCHS, '| batch:', BATCH_SIZE, '| fp16:', FP16, '| max_seq:', MAX_SEQ_LENGTH)

## 4) Load train / valid

Dữ liệu kỳ vọng có dạng `query` + `positive` trong `train_cleaned.jsonl` và `valid_cleaned.jsonl`.

In [ ]:
import json
from datasets import Dataset

def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(DATA_DIR / 'train_cleaned.jsonl')
valid_rows = load_jsonl(DATA_DIR / 'valid_cleaned.jsonl')
print('train:', len(train_rows), 'valid:', len(valid_rows))

train_ds = Dataset.from_list([{'anchor': r['query'], 'positive': r['positive']} for r in train_rows])
valid_ds = Dataset.from_list([{'anchor': r['query'], 'positive': r['positive']} for r in valid_rows])

## 5) Fine-tune E5-base với MultipleNegativesRankingLoss

Lưu ý: với E5, nên encode query và passage đúng quy ước `query:` / `passage:` khi evaluate.
Trong fine-tune, loss `MultipleNegativesRankingLoss` dùng positive pairs và các sample khác trong batch làm negative ngầm.

In [ ]:
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import SentenceTransformerTrainingArguments, BatchSamplers

model = SentenceTransformer(PRESET.base_model)
model.max_seq_length = MAX_SEQ_LENGTH
loss = MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir=str(MODELS_DIR / PRESET.name),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=PRESET.learning_rate,
    warmup_ratio=PRESET.warmup_ratio,
    fp16=FP16,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    save_strategy='epoch',
    eval_strategy='epoch',
    logging_steps=20,
    save_total_limit=2,
    run_name='e5-base-vi-embedding-colab',
    report_to=[],
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    loss=loss,
)

trainer.train()
model.save(str(FINAL_DIR))
print('Saved model to:', FINAL_DIR)

## 6) Evaluate pretrained vs fine-tuned

Chỉ số: Precision@10, Recall@10, MRR@10, NDCG@10.
Nếu muốn đúng chuẩn E5 hơn nữa, dùng prefix `query:` cho query và `passage:` cho corpus text.

In [ ]:
import numpy as np
import pandas as pd

def normalize(x):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-12)

def topk_indices(query_emb, corpus_emb, k=10):
    scores = query_emb @ corpus_emb.T
    idx = np.argpartition(-scores, kth=min(k, scores.shape[1]-1), axis=1)[:, :k]
    rows = np.arange(scores.shape[0])[:, None]
    top_scores = scores[rows, idx]
    order = np.argsort(-top_scores, axis=1)
    return idx[rows, order]

def metrics_at_k(retrieved, queries, labels, k=10):
    p_list, r_list, mrr_list, ndcg_list = [], [], [], []
    for q, hits in zip(queries, retrieved):
        rel = labels.get(q, set())
        if not rel:
            continue
        top_hits = hits[:k]
        flags = [1 if h in rel else 0 for h in top_hits]
        hit_count = sum(flags)
        p_list.append(hit_count / k)
        r_list.append(hit_count / len(rel))
        rr = next((1.0 / i for i, f in enumerate(flags, 1) if f), 0.0)
        mrr_list.append(rr)
        dcg = sum(f / np.log2(i + 2) for i, f in enumerate(flags))
        ideal = [1] * min(len(rel), k) + [0] * (k - min(len(rel), k))
        idcg = sum(f / np.log2(i + 2) for i, f in enumerate(ideal))
        ndcg_list.append(dcg / idcg if idcg > 0 else 0.0)
    return {
        'Precision@10': float(np.mean(p_list)) if p_list else 0.0,
        'Recall@10': float(np.mean(r_list)) if r_list else 0.0,
        'MRR@10': float(np.mean(mrr_list)) if mrr_list else 0.0,
        'NDCG@10': float(np.mean(ndcg_list)) if ndcg_list else 0.0,
    }

test_rows = load_jsonl(DATA_DIR / 'test_cleaned.jsonl')
labels_rows = json.loads((DATA_DIR / 'query_product_labels_cleaned.json').read_text(encoding='utf-8'))
labels = {r['query']: set(r['relevant_product_ids']) for r in labels_rows}

products_df = pd.read_csv(DATA_DIR / 'merged_products_vi_cleaned.csv')
products_df = products_df.dropna(subset=['product_id', 'searchable_text']).drop_duplicates('product_id')
product_ids = products_df['product_id'].astype(str).tolist()
corpus_texts = products_df['searchable_text'].astype(str).tolist()
query_texts = sorted({r['query'] for r in test_rows if r['query'] in labels})

# E5 convention
query_texts_e5 = [f'query: {q}' for q in query_texts]
corpus_texts_e5 = [f'passage: {t}' for t in corpus_texts]

def evaluate_model(model_name_or_path):
    m = SentenceTransformer(model_name_or_path)
    ce = normalize(m.encode(corpus_texts_e5, batch_size=128, show_progress_bar=True, convert_to_numpy=True))
    qe = normalize(m.encode(query_texts_e5, batch_size=128, show_progress_bar=True, convert_to_numpy=True))
    idx = topk_indices(qe, ce, k=10)
    retrieved = [[product_ids[i] for i in row] for row in idx]
    return metrics_at_k(retrieved, query_texts, labels, k=10)

pretrained_metrics = evaluate_model(PRESET.base_model)
finetuned_metrics = evaluate_model(str(FINAL_DIR))

result = {'preset': 'e5-base', 'pretrained': pretrained_metrics, 'finetuned': finetuned_metrics}
print(json.dumps(result, ensure_ascii=False, indent=2))
metrics_path = OUTPUT_EVAL_DIR / PRESET.metrics_filename
metrics_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved metrics:', metrics_path)

## 7) Gợi ý nâng cấp dữ liệu và training

- làm sạch query trùng lặp và gộp các biến thể ngôn ngữ
- tăng chất lượng positive pair
- bổ sung hard negatives
- thử hard mining + multi-stage fine-tune

In [ ]:
print('Fine-tune E5-base notebook ready.')